# Ensembl genes table extraction EDA

This notebook is useful for development as well as exploratory data analysis on the extracted tables.
It is currently automically executed and saved as part of exports using `papermill`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from bioregistry import normalize_prefix

from ensembl_genes import ensembl_genes

In [3]:
# parameters cell
species = "human"
release = "111"

In [4]:
# Parameters
species = "rat"
release = "116"


In [5]:
ensg = ensembl_genes.Ensembl_Gene_Queries(release=release, species=species)
ensg.connection_url

'mysql+mysqlconnector://anonymous@ensembldb.ensembl.org:3306/rattus_norvegicus_core_116_1'

In [6]:
database = ensg.database
database

'rattus_norvegicus_core_116_1'

## Extract data

## gene attrib counts

In [7]:
ensg.run_query("gene_attrib_counts").head(15)

,attrib_type_id,code,name,description,attrib_type_count,attrib_type_examples
0,142,GeneGC,Gene GC,Percentage GC content for this gene,43360,"51.01, 48.89, 34.18, 45.40, 49.21, 44.47, 37.8..."
1,559,added_seq_ann_pr_url,Added sequence annotation provider url,Source annotation provider url for the out of ...,0,None
2,127,cds_end_NF,CDS end not found,None,0,None
3,32,KnwnPCCount,protein_coding_KNOWN,Number of Known Protein Coding,0,None
4,90,bacend_well_nam,BACend well name,None,0,None
5,118,ensembl_name,Ensembl name,Name of equivalent Ensembl chromosome,0,None
6,565,samples_mutation,Number samples with mutation type,Number of samples with mutation type from the ...,0,None
7,520,proj_parent_g,projection parent gene,Stable identifier of the parent gene this gene...,0,None
8,358,PHIbase_mutant,PHI-base mutant,PHI-base phenotype of the mutants,0,None
9,560,artef_dupl,Artifactual duplication,Annotation on artifactual regions of the genom...,0,None


## genes

In [8]:
ensg.gene_df.head()

/home/runner/work/ensembl-genes/ensembl-genes/ensembl_genes/ensembl_genes.py:141: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(self._alt_allele_add_representative)


,ensembl_gene_id,ensembl_gene_version,gene_symbol,gene_symbol_source_db,gene_symbol_source_id,gene_biotype,ensembl_source,ensembl_created_date,ensembl_modified_date,coord_system_version,...,seq_region_start,seq_region_end,seq_region_strand,primary_assembly,lrg_gene_id,mhc,gene_description,gene_description_source_db,gene_description_source_id,ensembl_representative_gene_id
0,ENSRNOG00000000001,7,Arsj,RGD,1307640,protein_coding,ensembl,2009-07-29 15:36:02,2024-05-09 12:20:34,GRCr8,...,217449147,217529142,1,True,NaN,None,"arylsulfatase family, member J",RGD Symbol,1307640,ENSRNOG00000000001
1,ENSRNOG00000000007,9,Gad1,RGD,2652,protein_coding,ensembl,2009-07-29 15:36:02,2024-05-09 12:20:34,GRCr8,...,75777534,75818759,1,True,NaN,None,glutamate decarboxylase 1,RGD Symbol,2652,ENSRNOG00000000007
2,ENSRNOG00000000008,9,Alx4,RGD,1310201,protein_coding,ensembl,2009-07-29 15:36:02,2024-05-09 12:20:34,GRCr8,...,100067052,100103624,1,True,NaN,None,ALX homeobox 4,RGD Symbol,1310201,ENSRNOG00000000008
3,ENSRNOG00000000009,7,Tmco5b,RGD,1561237,protein_coding,ensembl,2009-07-29 15:36:02,2024-05-09 12:20:34,GRCr8,...,120519323,120537501,1,True,NaN,None,transmembrane and coiled-coil domains 5B,RGD Symbol,1561237,ENSRNOG00000000009
4,ENSRNOG00000000010,7,Cbln1,RGD,1562813,protein_coding,ensembl,2009-07-29 15:36:02,2024-05-09 12:20:34,GRCr8,...,35781952,35785960,1,True,NaN,None,cerebellin 1,RGD Symbol,1562813,ENSRNOG00000000010


In [9]:
# clone-based genes no longer get a symbol and are filled with the stable ID
# https://www.ensembl.info/2021/03/15/retirement-of-clone-based-gene-names/
ensg.gene_df.query("gene_symbol == ensembl_gene_id").head(2)

,ensembl_gene_id,ensembl_gene_version,gene_symbol,gene_symbol_source_db,gene_symbol_source_id,gene_biotype,ensembl_source,ensembl_created_date,ensembl_modified_date,coord_system_version,...,seq_region_start,seq_region_end,seq_region_strand,primary_assembly,lrg_gene_id,mhc,gene_description,gene_description_source_db,gene_description_source_id,ensembl_representative_gene_id
4678,ENSRNOG00000009738,7,ENSRNOG00000009738,None,None,protein_coding,ensembl,2009-07-29 15:36:02,2024-05-09 12:20:34,GRCr8,...,5556327,5599662,1,True,NaN,None,NaN,NaN,NaN,ENSRNOG00000009738
7069,ENSRNOG00000014455,5,ENSRNOG00000014455,None,None,protein_coding,ensembl,2009-07-29 15:36:02,2024-05-09 12:20:34,GRCr8,...,101889328,101891012,-1,True,NaN,None,NaN,NaN,NaN,ENSRNOG00000014455


In [10]:
# which external database the gene symbol derives from versus the ensembl source
pd.crosstab(
    ensg.gene_df.ensembl_source,
    ensg.gene_df.gene_symbol_source_db.fillna("missing (clone-based)"),
    margins=True,
)

gene_symbol_source_db,EntrezGene,MGI,RFAM,RGD,missing (clone-based),All
ensembl_source,,,,,,
RefSeq,0,0,0,36,1,37
ensembl,2,1,2,27126,16192,43323
All,2,1,2,27162,16193,43360


In [11]:
ensg.gene_df.coord_system.value_counts().head(10)

coord_system
primary_assembly    43360
Name: count, dtype: int64

In [12]:
ensg.gene_df.gene_biotype.value_counts().head(10)

gene_biotype
protein_coding          22016
lncRNA                  16149
snoRNA                   1988
snRNA                    1561
pseudogene                402
IG_V_gene                 324
rRNA                      279
miRNA                     245
TR_V_gene                 180
processed_pseudogene       45
Name: count, dtype: int64

In [13]:
pd.crosstab(ensg.gene_df.coord_system, ensg.gene_df.primary_assembly, margins=True)

primary_assembly,False,True,All
coord_system,,,
primary_assembly,29,43331,43360
All,29,43331,43360


In [14]:
ensg.gene_df.mhc.value_counts()

Series([], Name: count, dtype: int64)

In [15]:
len(ensg.gene_df)

43360

## alternative gene alleles

Related:

- [OTP: Origin of genes_with_non_reference_ensembl_ids.tsv](https://github.com/opentargets/platform/issues/702)
- [biostars: map between different assemblies of one ensembl release](https://www.biostars.org/p/143956/)
- using `attrib_type.code = "non_ref"` for `primary_assembly` doesn't appear to return any results

In [16]:
ensg.representative_gene_df.head()

,rs_allele_group,ensembl_gene_id,gene_symbol,ensembl_created_date,seq_region,primary_assembly,alt_allele_group_id,alt_allele_attrib,alt_allele_is_representative,ensembl_representative_gene_id,is_representative_gene
0,1700001E04Rikl,ENSRNOG00000033341,1700001E04Rikl,2005-03-02 00:00:00,9,True,None,None,False,ENSRNOG00000033341,True
1,1700001E04Rikl,ENSRNOG00000084146,1700001E04Rikl,2024-05-09 12:20:34,9,True,None,None,False,ENSRNOG00000033341,False
2,1700001K19Rikl,ENSRNOG00000007184,1700001K19Rikl,2009-07-29 15:36:02,6,True,None,None,False,ENSRNOG00000007184,True
3,1700006A11Rikl,ENSRNOG00000024928,1700006A11Rikl,2009-07-29 15:36:02,2,True,None,None,False,ENSRNOG00000024928,True
4,1700009N14Rikl,ENSRNOG00000031013,1700009N14Rikl,2005-03-02 00:00:00,5,True,None,None,False,ENSRNOG00000031013,True


In [17]:
# looks like non_ref isn't set for human genes
query = """
SELECT *
FROM gene_attrib
LEFT JOIN attrib_type
  ON gene_attrib.attrib_type_id = attrib_type.attrib_type_id
WHERE attrib_type.code = "non_ref"
LIMIT 5
"""
pd.read_sql(sql=query, con=ensg.connection_url)

,gene_id,attrib_type_id,value,attrib_type_id,code,name,description


In [18]:
ensg.representative_gene_df.alt_allele_attrib.value_counts()

Series([], Name: count, dtype: int64)

In [19]:
ensg.representative_gene_df.query(
    "ensembl_gene_id != ensembl_representative_gene_id"
).head(2)

,rs_allele_group,ensembl_gene_id,gene_symbol,ensembl_created_date,seq_region,primary_assembly,alt_allele_group_id,alt_allele_attrib,alt_allele_is_representative,ensembl_representative_gene_id,is_representative_gene
1,1700001E04Rikl,ENSRNOG00000084146,1700001E04Rikl,2024-05-09 12:20:34,9,True,None,None,False,ENSRNOG00000033341,False
346,Abcb1b,ENSRNOG00000066042,Abcb1b,2021-02-26 12:35:27,4,True,None,None,False,ENSRNOG00000008012,False


# replaced ID converter

A single `old_stable_id` can map to multiple `new_stable_id`. For example, `ENSG00000152006`

https://uswest.ensembl.org/Homo_sapiens/Tools/IDMapper/Results?tl=AzhM62SpkvdiLC4H-6808613

Requested ID | Matched ID(s) | Releases
-- | -- | --
ENSG00000152006 | ENSG00000196273 | 26: ENSG00000196273.1
ENSG00000152006 | ENSG00000197016 | 26: ENSG00000197016.1
ENSG00000152006 | ENSG00000196239 | 26: ENSG00000196239.1

In [20]:
ensg.old_to_new_df.head(2)

,old_ensembl_gene_id,new_ensembl_gene_id
0,ENSRNOG00000000132,ENSRNOG00000031425
1,ENSRNOG00000000194,ENSRNOG00000031589


In [21]:
# some ensembl genes replaced by many new ensembl genes
ensg.old_to_new_df.old_ensembl_gene_id.value_counts().head(2)

old_ensembl_gene_id
ENSRNOG00000034138    202
ENSRNOG00000062504    157
Name: count, dtype: int64

In [22]:
# example
ensg._update_ensembl_gene("ENSG00000152006")

{'ENSG00000152006'}

In [23]:
ensg.old_to_newest_df.head(2)

,old_ensembl_gene_id,newest_ensembl_gene_id,is_current
0,ENSRNOG00000000132,ENSRNOG00000031425,True
1,ENSRNOG00000000194,ENSRNOG00000031589,False


In [24]:
len(ensg.old_to_newest_df)

94320

In [25]:
ensg.old_to_newest_df.is_current.value_counts()

is_current
True     87285
False     7035
Name: count, dtype: int64

## omni-updater

The omni-updater dataset is designed to convert ensembl gene IDs from input data to the current, representative ensembl_gene_ids for this ensembl release. It assumes:

- users want to update outdated genes with their replacements
- users want a dataset of representative genes only, and want to convert alternative alleles to representative genes

An inner join of a dataset with `update_df` on `input_ensembl_gene_id` will do the following:

- produce output ensembl_gene_ids that are current and representatives
- update outdated genes with their current identifiers. Outdated genes with no current replacement will be removed by the inner join.
- update alternative gene alleles with their representatives
- genes that are already represenative and current will map to themselves

In [26]:
ensg.update_df.head(2)

input_ensembl_gene_id    ensembl_gene_id  input_current  input_representative
   ENSRNOG00000029508 ENSRNOG00000029508           True                  True
   ENSRNOG00000029508 ENSRNOG00000029508          False                 False
   ENSRNOG00000031896 ENSRNOG00000031896           True                  True
   ENSRNOG00000031896 ENSRNOG00000031896          False                 False
   ENSRNOG00000034061 ENSRNOG00000034061           True                  True
   ENSRNOG00000034061 ENSRNOG00000034061          False                 False


,input_ensembl_gene_id,ensembl_gene_id,input_current,input_representative,input_maps_to_n_genes,n_inputs_map_to_gene
0,ENSRNOG00000000001,ENSRNOG00000000001,True,True,1,1
1,ENSRNOG00000000007,ENSRNOG00000000007,True,True,1,1


In [27]:
ensg.update_df.sort_values("input_maps_to_n_genes", ascending=False).head(2)

,input_ensembl_gene_id,ensembl_gene_id,input_current,input_representative,input_maps_to_n_genes,n_inputs_map_to_gene
25147,ENSRNOG00000036156,ENSRNOG00000090384,False,True,175,160
25148,ENSRNOG00000036156,ENSRNOG00000090735,False,True,175,161


In [28]:
ensg.update_df.sort_values("n_inputs_map_to_gene", ascending=False).head(2)

,input_ensembl_gene_id,ensembl_gene_id,input_current,input_representative,input_maps_to_n_genes,n_inputs_map_to_gene
42704,ENSRNOG00000090735,ENSRNOG00000090735,True,True,1,161
42734,ENSRNOG00000090765,ENSRNOG00000090765,True,True,1,161


In [29]:
(ensg.update_df.input_maps_to_n_genes == 1).mean()

np.float64(0.33905901641856223)

In [30]:
ensg.update_df.query("ensembl_gene_id == 'ENSG00000256263'")

,input_ensembl_gene_id,ensembl_gene_id,input_current,input_representative,input_maps_to_n_genes,n_inputs_map_to_gene


In [31]:
print(
    f"The omni-updater contains {len(ensg.update_df):,} rows for mapping "
    f"{ensg.update_df.input_ensembl_gene_id.nunique():,} input genes to "
    f"{ensg.update_df.ensembl_gene_id.nunique():,} current, representative genes."
)

The omni-updater contains 130,523 rows for mapping 48,813 input genes to 42,905 current, representative genes.


In [32]:
# https://useast.ensembl.org/Homo_sapiens/Tools/IDMapper/Results?tl=P45VLMbogubpI0QA-6815464
ensg.update_df.query("input_ensembl_gene_id == 'ENSG00000201456'").head(3)

,input_ensembl_gene_id,ensembl_gene_id,input_current,input_representative,input_maps_to_n_genes,n_inputs_map_to_gene


## cross-refrences (xrefs)

In [33]:
ensg.xref_df.head()

,ensembl_representative_gene_id,ensembl_gene_id,gene_symbol,xref_source,xref_accession,xref_label,xref_description,xref_info_type,xref_linkage_annotation,xref_curie
0,ENSRNOG00000000001,ENSRNOG00000000001,Arsj,ArrayExpress,ENSRNOG00000000001,ENSRNOG00000000001,None,DIRECT,None,arrayexpress:ENSRNOG00000000001
1,ENSRNOG00000000001,ENSRNOG00000000001,Arsj,EntrezGene,311013,Arsj,"arylsulfatase family, member J",DEPENDENT,None,ncbigene:311013
2,ENSRNOG00000000001,ENSRNOG00000000001,Arsj,HGNC,HGNC:26286,ARSJ,arylsulfatase family member J,NONE,None,hgnc:26286
12,ENSRNOG00000000001,ENSRNOG00000000001,Arsj,RGD,1307640,Arsj,"arylsulfatase family, member J",DIRECT,None,rgd:1307640
13,ENSRNOG00000000001,ENSRNOG00000000001,Arsj,RGD,15003202,AABR07013255.1,,DIRECT,None,rgd:15003202


In [34]:
# datasets where there are ensembl_gene_id-xref_source-xref_accession pairs might not be distinct
xref_dup_df = ensg.xref_df[
    ensg.xref_df.duplicated(
        subset=["ensembl_gene_id", "xref_source", "xref_accession"], keep=False
    )
]
xref_dup_df.xref_source.value_counts()

xref_source
MGI    2
Name: count, dtype: int64

In [35]:
# xref sources versus info_types
df = pd.crosstab(ensg.xref_df.xref_source, ensg.xref_df.xref_info_type, margins=True)
df["bioregistry_prefix"] = (
    df.index.to_series().replace(ensg._xref_prefix_updater).map(normalize_prefix)
)
df

xref_info_type,DEPENDENT,DIRECT,NONE,PROJECTION,SEQUENCE_MATCH,All,bioregistry_prefix
xref_source,,,,,,,
ArrayExpress,0,43360,0,0,0,43360,arrayexpress
EntrezGene,37897,0,0,0,0,37897,ncbigene
FlyBaseName_gene,0,0,1,0,0,1,None
HGNC,0,0,15210,0,0,15210,hgnc
MGI,0,0,2812,1,0,2813,mgi
RFAM,0,3947,0,0,0,3947,rfam
RGD,602,28389,0,0,0,28991,rgd
Reactome_gene,0,86033,0,0,0,86033,reactome
SGD_GENE,0,0,1,0,0,1,None


## Gene Ontology xrefs

In [36]:
ensg.xref_go_df.head(3)

,ensembl_gene_id,go_id,go_label,go_evidence_codes,xref_info_types,xref_info_texts,ensembl_transcript_ids,ensembl_representative_gene_id
0,ENSRNOG00000000001,GO:0008484,sulfuric ester hydrolase activity,None,DEPENDENT,,ENSRNOT00000055633,ENSRNOG00000000001
1,ENSRNOG00000000001,GO:0015629,actin cytoskeleton,IEA,PROJECTION,from homo_sapiens translation ENSP00000320219,ENSRNOT00000055633,ENSRNOG00000000001
2,ENSRNOG00000000007,GO:0004351,glutamate decarboxylase activity,IEA,PROJECTION,"from homo_sapiens translation ENSP00000350928,...",ENSRNOT00000087134,ENSRNOG00000000007


In [37]:
# GO terms for CCR5
# compare to http://useast.ensembl.org/Homo_sapiens/Gene/Ontologies/molecular_function?g=ENSG00000160791
# Using sort_values() to handle potential None values in go_label
ensg.xref_go_df.query(
    "ensembl_gene_id == 'ENSG00000160791'"
).go_label.sort_values().tolist()

[]

In [38]:
# Check for GO terms with missing labels
# https://github.com/related-sciences/ensembl-genes/issues/28
missing_go_label_df = ensg.xref_go_df[ensg.xref_go_df.go_label.isna()]
if not missing_go_label_df.empty:
    print(f"Found {len(missing_go_label_df):,} GO annotations with missing go_label")
    display(missing_go_label_df.head(10))
else:
    print("All GO terms have labels")

Found 5,559 GO annotations with missing go_label


,ensembl_gene_id,go_id,go_label,go_evidence_codes,xref_info_types,xref_info_texts,ensembl_transcript_ids,ensembl_representative_gene_id
52,ENSRNOG00000000010,GO:1900454,None,IEA,PROJECTION,from homo_sapiens translation ENSP00000219197,ENSRNOT00000000011,ENSRNOG00000000010
66,ENSRNOG00000000012,GO:1902037,None,IEA,PROJECTION,from mus_musculus translation ENSMUSP00000086511,ENSRNOT00000000013,ENSRNOG00000000012
173,ENSRNOG00000000054,GO:1900226,None,IEA,PROJECTION,"from homo_sapiens translation ENSP00000247706,...",ENSRNOT00000096877,ENSRNOG00000000054
200,ENSRNOG00000000065,GO:7770030,None,IEA,PROJECTION,from homo_sapiens translation ENSP00000420295,ENSRNOT00000000077,ENSRNOG00000000065
207,ENSRNOG00000000066,GO:1990756,None,IEA,PROJECTION,from homo_sapiens translation ENSP00000355827,ENSRNOT00000000074,ENSRNOG00000000066
256,ENSRNOG00000000075,GO:0140003,None,"IDA,ISO",DIRECT,"RGD,UniProt","ENSRNOT00000000086,ENSRNOT00000155846",ENSRNOG00000000075
281,ENSRNOG00000000082,GO:0140234,None,IEA,PROJECTION,from homo_sapiens translation ENSP00000308944,ENSRNOT00000106667,ENSRNOG00000000082
282,ENSRNOG00000000082,GO:0140658,None,IEA,PROJECTION,from homo_sapiens translation ENSP00000308944,ENSRNOT00000106667,ENSRNOG00000000082
283,ENSRNOG00000000082,GO:0141119,None,IEA,PROJECTION,from homo_sapiens translation ENSP00000308944,ENSRNOT00000106667,ENSRNOG00000000082
284,ENSRNOG00000000082,GO:0160225,None,IEA,PROJECTION,from homo_sapiens translation ENSP00000308944,ENSRNOT00000106667,ENSRNOG00000000082


## lrg xrefs

In [39]:
ensg.xref_lrg_df.head(2)

,ensembl_gene_id,lrg_gene_id


In [40]:
len(ensg.xref_lrg_df)

0

### ncbigene xrefs

In [41]:
ensg.xref_ncbigene_df.head()

,ensembl_representative_gene_id,ncbigene_id,gene_symbol,ncbigene_symbol
1,ENSRNOG00000000001,311013,Arsj,Arsj
18,ENSRNOG00000000007,24379,Gad1,Gad1
32,ENSRNOG00000000008,296511,Alx4,Alx4
40,ENSRNOG00000000009,366158,Tmco5b,Tmco5b
46,ENSRNOG00000000010,498922,Cbln1,Cbln1


In [42]:
# ensembl gene mapped to by multiple ncbigenes
ensg.xref_ncbigene_df.ensembl_representative_gene_id.value_counts().head(3)

ensembl_representative_gene_id
ENSRNOG00000000001    1
ENSRNOG00000000007    1
ENSRNOG00000000008    1
Name: count, dtype: int64

In [43]:
(
    len(ensg.xref_ncbigene_df),
    ensg.xref_ncbigene_df.ensembl_representative_gene_id.duplicated().sum(),
)

(23316, np.int64(0))

In [44]:
# ncbigene mapped to by multiple ensembl genes, likely due to alt gene alleles
ensg.xref_ncbigene_df.ncbigene_id.value_counts().head(3)

ncbigene_id
100364835    3
362588       3
120097727    2
Name: count, dtype: int64

In [45]:
len(ensg.xref_ncbigene_df), ensg.xref_ncbigene_df.ncbigene_id.duplicated().sum()

(23316, np.int64(11))

In [46]:
# ensg.xref_ncbigene_df.query("ensembl_representative_gene_id == 'ENSG00000231500'")
# ensg.xref_ncbigene_df.query("ncbigene_id == '51206'")

In [47]:
repr_ensembl_gene_ids = set(ensg.gene_df.ensembl_representative_gene_id)
len(repr_ensembl_gene_ids)

42905

In [48]:
# many of these genes should probably be alternative alleles rather than representative
ensg.gene_df.query(
    "not primary_assembly and ensembl_gene_id==ensembl_representative_gene_id"
)

,ensembl_gene_id,ensembl_gene_version,gene_symbol,gene_symbol_source_db,gene_symbol_source_id,gene_biotype,ensembl_source,ensembl_created_date,ensembl_modified_date,coord_system_version,...,seq_region_start,seq_region_end,seq_region_strand,primary_assembly,lrg_gene_id,mhc,gene_description,gene_description_source_db,gene_description_source_id,ensembl_representative_gene_id
20198,ENSRNOG00000062265,3,AABR07034262.1,RGD,15014745,lncRNA,ensembl,2017-01-18 14:25:58,2024-05-09 12:20:34,GRCr8,...,910449,989594,1,False,NaN,None,NaN,NaN,NaN,ENSRNOG00000062265
20623,ENSRNOG00000063513,2,ENSRNOG00000063513,None,None,lncRNA,ensembl,2021-02-26 12:35:27,2024-05-09 12:20:34,GRCr8,...,1224634,1231178,1,False,NaN,None,NaN,NaN,NaN,ENSRNOG00000063513
26743,ENSRNOG00000074774,1,LOC134484554,RGD,402129324,lncRNA,ensembl,2024-05-09 12:20:34,2024-05-09 12:20:34,GRCr8,...,355420,450968,-1,False,NaN,None,NaN,NaN,NaN,ENSRNOG00000074774
26879,ENSRNOG00000074910,1,ENSRNOG00000074910,None,None,lncRNA,ensembl,2024-05-09 12:20:34,2024-05-09 12:20:34,GRCr8,...,181720,192666,-1,False,NaN,None,NaN,NaN,NaN,ENSRNOG00000074910
26996,ENSRNOG00000075027,1,ENSRNOG00000075027,None,None,lncRNA,ensembl,2024-05-09 12:20:34,2024-05-09 12:20:34,GRCr8,...,655686,833957,1,False,NaN,None,NaN,NaN,NaN,ENSRNOG00000075027
27786,ENSRNOG00000075817,1,ENSRNOG00000075817,None,None,lncRNA,ensembl,2024-05-09 12:20:34,2024-05-09 12:20:34,GRCr8,...,1239880,1271675,1,False,NaN,None,NaN,NaN,NaN,ENSRNOG00000075817
28831,ENSRNOG00000076862,1,ENSRNOG00000076862,None,None,lncRNA,ensembl,2024-05-09 12:20:34,2024-05-09 12:20:34,GRCr8,...,632225,722368,-1,False,NaN,None,NaN,NaN,NaN,ENSRNOG00000076862
28913,ENSRNOG00000076944,1,Idi2-ps1,RGD,9352023,protein_coding,ensembl,2024-05-09 12:20:34,2024-05-09 12:20:34,GRCr8,...,55886,58475,1,False,NaN,None,"isopentenyl-diphosphate delta isomerase 2, pse...",RGD Symbol,9352023,ENSRNOG00000076944
30029,ENSRNOG00000078060,1,ENSRNOG00000078060,None,None,lncRNA,ensembl,2024-05-09 12:20:34,2024-05-09 12:20:34,GRCr8,...,1198376,1203949,1,False,NaN,None,NaN,NaN,NaN,ENSRNOG00000078060
30122,ENSRNOG00000078153,1,ENSRNOG00000078153,None,None,protein_coding,ensembl,2024-05-09 12:20:34,2024-05-09 12:20:34,GRCr8,...,1372726,1380096,1,False,NaN,None,NaN,NaN,NaN,ENSRNOG00000078153
